In [19]:
import json
from dataclasses import asdict
from itertools import islice

import pandas as pd

from ireland_property_analysis.ingestion import (
    clean_listing_with_outcome,
    iter_raw_csv_rows,
    parse_raw_property_listing,
    summarize_ingestion_run,
)
from ireland_property_analysis.schema import CleanedPropertyListing
from ireland_property_analysis.utils import find_project_root

ROOT_DIR = find_project_root()
csv_path = ROOT_DIR / "sample-data" / "PPR-ALL.csv"

In [20]:
# use 2 rows as an example

rows = list(islice(iter_raw_csv_rows(csv_path), 2))

In [21]:
raw_listings = [parse_raw_property_listing(row) for row in rows]
outcomes = [clean_listing_with_outcome(listing) for listing in raw_listings]
report = summarize_ingestion_run(outcomes)
cleaned_records = [
    result.outcome
    for result in outcomes
    if isinstance(result.outcome, CleanedPropertyListing)
]

report, cleaned_records

(IngestionRunReport(total_records=2, successful_records=1, failed_records=1, errors={'2': PropertyRawDataValidationError('Error processing record at row 2: raw_area must not be None or empty')}),
 [CleanedPropertyListing(id='3', date_of_sale=datetime.date(2010, 3, 1), county='Laois', price=Decimal('209975.00000'), price_per_sq_m=Decimal('5525.657894736842105263157895'), description='New Dwelling house /Apartment', address='134 Ashewood Walk, Summerhill Lane, Portlaoise', is_vat_exclusive=True, size_description='greater than or equal to 38 sq metres and less than 125 sq metres', is_full_market_price=True)])

In [22]:
# write to parquet

output_dir = ROOT_DIR / "processed-data"
output_dir.mkdir(exist_ok=True)

cleaned_rcords_df = pd.DataFrame([asdict(record) for record in cleaned_records])
cleaned_rcords_df.to_parquet(output_dir / "cleaned_properties.parquet", index=False)

In [23]:
print(report)

IngestionRunReport(total_records=2, successful_records=1, failed_records=1, errors={'2': PropertyRawDataValidationError('Error processing record at row 2: raw_area must not be None or empty')})


In [24]:
report_data = asdict(report)
report_data["errors"] = {
    row_number: str(error)
    for row_number, error in report.errors.items()
}

with open(output_dir / "ingestion_report.json", "w") as report_file:
    json.dump(report_data, report_file, indent=4)